# Projet 6 - Initiez-vous au MLOps (partie 1/2)


**Objectif du projet**  
Prédire la probabilité de défaut de paiement d'un client (TARGET = 1)  
→ Problème de **classification binaire déséquilibrée**

**Approche MLOps visée dans cette première partie**  
- Suivi systématique des expériences avec **MLflow**  
- Comparaison de plusieurs modèles / jeux de features  
- Versionning des modèles  



Date : Janvier 2026  
Auteur : Joannes Landy

## Étape 1 - Préparez, nettoyez et enrichissez les données

**Préparation des données**

**Objectif**  
Constituer un jeu de données propre, fusionné et enrichi à partir des différentes sources, prêt pour l'entraînement d'un modèle de scoring de crédit.

### Exploration des données bruts

Description des champs présents dans les données bruts.

Précision sur le type de donnée, les valeurs statistiques de base, min, max, moyenne, quantiles, valeurs null, valeurs identiques.

In [2]:
# Importations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import mlflow
import os

import logging
logger = logging.getLogger(__name__)


import warnings
warnings.filterwarnings('ignore')


# Log in MlFlow, need mlflow running: mlflow ui
mlflow.set_experiment("Projet 06 - OpenClassrooms")


# Chargement des fichiers du repertoire data_path, création des dataframes

def csv_to_dataframe(data_path):
    """ list csv files in data_path and return dic with all dataframes """
    dataframes = {}
    files = list(Path(data_path).glob("*.csv"))
    
    message = f"{'name'.ljust(25)}\tcolonnes\tlignes\t\tdoublons"
    logger.info(message)
    
    for file in files:
    
        dataframe_name = file.name.replace(".csv", "")
        df = pd.read_csv(os.path.join(data_path, file.name))
        dataframes[dataframe_name] = df
        df.duplicated().sum()
        message = f"{dataframe_name.ljust(25)}\t{df.shape[1]}\t\t{df.shape[0]}\t\t{df.duplicated().sum()}"
        logger.info(message)

        cwd = Path.cwd()

        mlflow.log_artifact(os.path.join(data_path, file.name), artifact_path='test_projet_05')
    print('-----------------')
    return dataframes
    

df_brut = csv_to_dataframe("dataset")
      



ImportError: cannot import name 'service' from 'google.protobuf' (/home/joannes/Documents/openclassroom/06_Initiez_vous_au_MLOps_1-2/openclassrooms_projet06/.venv/lib/python3.13/site-packages/google/protobuf/__init__.py)

In [ ]:
# Création d'un dataframe de description des données
def describe_field(df):
    df_field_describ = pd.DataFrame({
        "name": df.columns,
        "dtype": df.dtypes.values,
        "unique": df.nunique(),
        "notna": df.notnull().sum().values,
        "null": df.isna().sum().values,
    })
    # Ajout des descriptions données par la fonction describe() des champs numérique
    df_describe_numeric = df.describe().T.reset_index().drop(columns=['count'])
    df_field_describ = df_field_describ.merge(df_describe_numeric, left_on='name', right_on='index', how='left').drop(columns=['index'])
    return df_field_describ

# Display description field
def display_describe_field(df_brut):
    """ Display the dataframe description  
    Input: {dataframe_name: dataframe, ...}
    Output: Jupyter display
    """
    for df_name, df_data in df_brut.items():
        df_field_describ =  describe_field(df_data)
        print('Dataframe shape: ', df_name, df_data.shape)
        display(df_field_describ)

# Save description field in excel for reading
def excel_describ(df_brut):
    """ describ the dataframe format and save the result in excel format 
    Input: {dataframe_name: dataframe, ...}
    Output: excel file
    """
    with pd.ExcelWriter("dataframes_fields_analyse.xlsx") as writer:
        for df_name, df_data in df_brut.items():
            df = describe_field(df_data)
            print('Dataframe shape: ', df_name, df_data.shape)
            display(df)
            df.to_excel(writer, sheet_name=df_name, index=False)

excel_describ(df_brut)


### Analyse de la distribution des données bruts

Visualisation des histogrammes

Recherche des outliers

In [ ]:
# Visualisation de la distribution des variables
column_index = ['SK_ID_BUREAU', 'SK_ID_PREV', 'SK_ID_CURR', 'SK_DPD', 'SK_DPD_DEF']

def view_histogram(df_brut, column_index):
    """ View histogram """
    for df_name, df_data in df_brut.items():

        nb_column = df_data.shape[1]
        nb_graph_y = 5
        nb_graph_x = int(nb_column / nb_graph_y) + 1
        

        fig, axs = plt.subplots(nb_graph_x, nb_graph_y, figsize=(20, nb_graph_x * 5))
        axs = axs.ravel()
        
        for i, column in enumerate(df_data.columns):
            axs[i].set_title(column)
            if column in column_index:
                continue
                
            try:    
                axs[i].hist(df_data[column], bins=50)
            except:
                continue
        print('Fichier: ', df_name)
        plt.show()
        

view_histogram(df_brut, column_index)

### Analyse de la variable cible

In [ ]:

plt.figure(figsize=(7, 5))
df_train = df_brut['application_train']

sns.countplot(data=df_train, x='TARGET', palette='viridis')
plt.title('Distribution de la variable cible (TARGET)', fontsize=14)
plt.xlabel('TARGET (1 = défaut de paiement)')
plt.ylabel('Nombre d\'observations')

total = len(df_train)
for p in plt.gca().patches:
    percentage = f'{100 * p.get_height() / total:.2f}%'
    plt.gca().annotate(percentage, (p.get_x() + p.get_width() / 2., p.get_height()),
                       ha='center', va='center', fontsize=11, xytext=(0, 8),
                       textcoords='offset points')

plt.show()

print("Répartition de la cible :")
print(df_train['TARGET'].value_counts(normalize=True)
      .mul(100)
      .round(2)
      .astype(str) + " %")


### Prétraitement des données

- ajout des features
- modification et adaptation des données


In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, make_scorer, accuracy_score
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import confusion_matrix, classification_report


from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline


import time


def merge_datasets(df_brut, target="TARGET"):
    """ Merge multiple dataframe into one 
    return: dataframe 
    """

    # Select and merge dataframe
    merged_df = df_brut['application_train']
    
    # Split X and y
    X = merged_df.drop(columns=[target])
    y = merged_df[target]
    return X, y

def drop_inconsistent_columns(df_preparation):
    """ Check and drop the consistencs of columns
    return: dataframe 
    """
    df_result = df_preparation.copy()
    return df_result
    
def drop_uninformative_columns(df_preparation):
    """ Check and drop uninformative_columns
        return: dataframe 
    """
    df_result = df_preparation.copy()
    return df_result
    
def split_test_train(dataframe, target, random_state, test_size=0.2):
    """ Split the dataset into train/test dataset
    
    """

    # Séparer les features (X) et la cible (y)
    X = dataframe.drop(columns=[target])
    y = dataframe[target]
    
    # Séparation train/test (80% train, 20% test), stratification des populations
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=y)    
    
    return X_train, X_test, y_train, y_test

    
# Création du jeu de test/train
random_state = 42
target = 'TARGET'

X, y = merge_datasets(df_brut, target)



## Étape 2 - Traquez les expérimentations avec MLFlow

Objectifs:

Des runs visibles dans l’UI MLflow avec les paramètres testés et les scores obtenus.

In [ ]:

# -----------------------------
# 2. Identifier colonnes numériques et catégorielles
# -----------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Features numériques : {numeric_features}")
print(f"Features catégorielles : {categorical_features}")

# -----------------------------
# 3. Créer le préprocesseur
# -----------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

# -----------------------------
# 4. Créer le pipeline complet
# -----------------------------
model = RandomForestClassifier(n_estimators=100, random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", model)
])

# -----------------------------
# 5. Séparer train / test
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,          # classification équilibrée
    random_state=42
)

# -----------------------------
# 6. Entraîner le pipeline
# -----------------------------
while mlflow.active_run():
    mlflow.end_run()

mlflow.sklearn.autolog()
pipeline.fit(X_train, y_train)

# -----------------------------
# 7. Évaluer
# -----------------------------
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n✅ Précision sur le test : {accuracy:.4f}")
print("\nRapport de classification :\n")
print(classification_report(y_test, y_pred))


    

In [ ]:


# Liste des modèles pour itération
models = {
    'dummy_model': DummyClassifier(),
    'linear_model': LogisticRegression(),
    'rf_model': RandomForestClassifier(),

}

# Liste des parametres à tester
param_grids = {
    'dummy_model': {},
    'linear_model': {
        'random_state': [random_state],
        'max_iter': [100, 1000],
        'class_weight': ['balanced'],
    },
    'rf_model': {
        'random_state': [random_state],
        'class_weight': ['balanced'],
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7],
        #'min_samples_split': [2, 5, 10],
        #'min_samples_leaf': [1, 2, 4]
    },


}

# chercher et Stocker les meilleurs modèles

def search_bestmodel(models, param_grids, X, y):
    """ Use GridSearchCV to find best parameter, return the best model"""
    best_models = {}
    for name, model in models.items():
        print(f"\nEntraînement de {name}...")
        
        numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
        categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


        preprocessor = ColumnTransformer(transformers=[
                ("num", StandardScaler(), numeric_features),
                ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
                ])
        pipeline = Pipeline(steps=[
                ("preprocessor", preprocessor),
                ("classifier", model)
                ])
        
        # GridSearchCV
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[name],
            scoring='f1',
            cv=5,                  # validation croisée à 5 folds
            n_jobs=-1,             # utiliser tous les cœurs
            verbose=1              # afficher la progression
        )
        
        # Ajuster sur les données d'entraînement
        grid.fit(X, y)
        
        # Sauvegarder le meilleur modèle
        best_models[name] = grid.best_estimator_
        
        print(f"Meilleurs paramètres : {grid.best_params_}")
        print(f"Meilleur score F1 : {grid.best_score_:.4f}")
    
    return best_models

best_models = search_bestmodel(models, param_grids, X, y)

## Étape 3 - Modélisez et expérimentez avec plusieurs algorithmes

Objectifs:
Un ou plusieurs modèles entraînés, avec validation croisée et premières métriques d’évaluation.

##  Étape 4 - Optimisez les hyper paramètres et le seuil métier

Objectifs:
Un modèle avec hyperparamètres optimisés et seuil métier ajusté.

